# 🎬 JIT Reel Enhancer (free GPU)
**What it does:** watches your Google Drive folder **JIT Reels**. Every reel you send from the app is made sharper and bigger with AI (Real-ESRGAN), saved in **JIT Reels › Enhanced**, and then the original reel is deleted, so only the enhanced reel stays.

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU** (only the first time).
2. **Runtime → Run all** (or Ctrl+F9). Allow Google Drive when asked.
3. Keep sending reels from the app. This page keeps working even with the phone screen off, until the free Colab session ends.

If the session stops, just **Run all** again: reels that are already done are not done twice.

In [ ]:
#@title ⚙️ Settings
SCALE = 2  #@param [2, 4] {type:"raw"}
SMOOTH_60FPS = False  #@param {type:"boolean"}
MODEL = "general"  #@param ["general", "fast"]
DELETE_ORIGINAL = True  #@param {type:"boolean"}
WATCH_MINUTES = 120  #@param {type:"integer"}
FOLDER = "JIT Reels"  #@param {type:"string"}
# SCALE 2 = 720p reel → 1440p (faster) · SCALE 4 = 4K (much slower)
# SMOOTH_60FPS = makes 30 fps reels 60 fps (slower)
# MODEL general = best for game/real video · fast = about 2x faster, a little softer
# WATCH_MINUTES = how long to keep waiting for new reels after the last one

In [ ]:
#@title 1️⃣ Setup: GPU, Google Drive, AI model
import os, sys, time, json, glob, subprocess, urllib.request
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Menu Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all again.")
DEV = torch.device("cuda")
from google.colab import drive
drive.mount("/content/drive")
SRC = f"/content/drive/MyDrive/{FOLDER}"
OUT = os.path.join(SRC, "Enhanced")
WORK = "/content/work"
os.makedirs(SRC, exist_ok=True); os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)

# Real-ESRGAN "compact" network (same as the official SRVGGNetCompact), written here so no extra package can break
class SRVGGNetCompact(nn.Module):
    def __init__(self, num_feat=64, num_conv=32, upscale=4):
        super().__init__()
        self.upscale = upscale
        body = [nn.Conv2d(3, num_feat, 3, 1, 1), nn.PReLU(num_parameters=num_feat)]
        for _ in range(num_conv):
            body += [nn.Conv2d(num_feat, num_feat, 3, 1, 1), nn.PReLU(num_parameters=num_feat)]
        body += [nn.Conv2d(num_feat, 3 * upscale * upscale, 3, 1, 1)]
        self.body = nn.ModuleList(body)
        self.upsampler = nn.PixelShuffle(upscale)
    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        return self.upsampler(out) + F.interpolate(x, scale_factor=self.upscale, mode="nearest")

MODELS = {
    "general": ("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth", 32),
    "fast":    ("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth", 16),
}
url, nconv = MODELS.get(MODEL, MODELS["general"])
wpath = "/content/" + url.rsplit("/", 1)[1]
if not os.path.exists(wpath):
    urllib.request.urlretrieve(url, wpath)
state = torch.load(wpath, map_location="cpu")
state = state.get("params_ema", state.get("params", state))
NET = SRVGGNetCompact(num_conv=nconv)
NET.load_state_dict(state, strict=True)
NET = NET.eval().half().to(DEV)

@torch.no_grad()
def _run(x):
    return NET(x)

@torch.no_grad()
def upscale_frames(frames, out_w, out_h):
    """frames: list of HxWx3 uint8 RGB → list of out_h x out_w x 3 uint8 RGB"""
    x = torch.from_numpy(np.stack(frames)).to(DEV).permute(0, 3, 1, 2).half().div_(255)
    try:
        y = _run(x)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        y = torch.cat([_tiled(x[i:i + 1]) for i in range(x.shape[0])])
    if y.shape[-1] != out_w or y.shape[-2] != out_h:
        y = F.interpolate(y.float(), size=(out_h, out_w), mode="area")
    y = y.clamp_(0, 1).mul_(255).round_().to(torch.uint8).permute(0, 2, 3, 1).contiguous().cpu().numpy()
    return [f for f in y]

@torch.no_grad()
def _tiled(x, tile=256, pad=16):
    # for very big frames: do the picture in pieces
    _, _, h, w = x.shape
    s = NET.upscale
    out = torch.zeros((1, 3, h * s, w * s), dtype=x.dtype, device=x.device)
    for y0 in range(0, h, tile):
        for x0 in range(0, w, tile):
            y1, x1 = min(y0 + tile, h), min(x0 + tile, w)
            ya, xa, yb, xb = max(y0 - pad, 0), max(x0 - pad, 0), min(y1 + pad, h), min(x1 + pad, w)
            o = _run(x[:, :, ya:yb, xa:xb])
            out[:, :, y0 * s:y1 * s, x0 * s:x1 * s] = o[:, :, (y0 - ya) * s:(y0 - ya + y1 - y0) * s, (x0 - xa) * s:(x0 - xa + x1 - x0) * s]
    return out

print(f"✅ Ready · GPU: {torch.cuda.get_device_name(0)} · model: {MODEL} · {SCALE}x · watching Drive › {FOLDER}")

In [ ]:
#@title 2️⃣ Enhancer
import os, time, glob, json, subprocess, cv2
VIDEO_EXT = (".mp4", ".webm", ".mov", ".mkv")

def probe(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "stream=codec_type,width,height:format=duration", "-of", "json", path], capture_output=True, text=True)
    j = json.loads(r.stdout or "{}")
    v = next((s for s in j.get("streams", []) if s.get("codec_type") == "video"), {})
    return {"dur": float(j.get("format", {}).get("duration") or 0), "w": int(v.get("width") or 0), "h": int(v.get("height") or 0),
            "audio": any(s.get("codec_type") == "audio" for s in j.get("streams", []))}

def _even(x):
    return int(x) // 2 * 2

def enhance(src):
    """one reel → OUT/<name>.mp4 (checked), then the original is deleted"""
    base = os.path.splitext(os.path.basename(src))[0]
    final = os.path.join(OUT, base + ".mp4")
    fps = 60 if SMOOTH_60FPS else 30
    tin, tout = os.path.join(WORK, "in.mp4"), os.path.join(WORK, "out.mp4")
    for p in (tin, tout):
        if os.path.exists(p): os.remove(p)
    info = probe(src)
    if info["dur"] <= 0 or not info["w"]:
        raise RuntimeError("cannot read this video")
    # 1) steady frame rate (phone recordings change it), optional 60 fps
    vf = f"minterpolate=fps={fps}:mi_mode=mci:mc_mode=aobmc:vsbmc=1" if SMOOTH_60FPS else f"fps={fps}"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", src, "-vf", vf, "-an", "-c:v", "libx264", "-preset", "veryfast", "-crf", "12", "-pix_fmt", "yuv420p", tin], check=True)
    cap = cv2.VideoCapture(tin)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    ow, oh = _even(w * SCALE), _even(h * SCALE)
    batch = 2 if w * h <= 1_000_000 else 1
    # 2) AI upscale every frame, straight into the encoder; sound from the original reel
    enc = subprocess.Popen(["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{ow}x{oh}", "-r", str(fps), "-i", "-",
                            "-i", src, "-map", "0:v:0", "-map", "1:a:0?", "-c:v", "libx264", "-preset", "medium", "-crf", "17", "-pix_fmt", "yuv420p",
                            "-c:a", "aac", "-b:a", "192k", "-shortest", "-movflags", "+faststart", tout], stdin=subprocess.PIPE)
    done, t0, shown, buf = 0, time.time(), -1, []
    try:
        while True:
            ok, frame = cap.read()
            if ok:
                buf.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if buf and (len(buf) >= batch or not ok):
                for f in upscale_frames(buf, ow, oh):
                    enc.stdin.write(f.tobytes())
                done += len(buf); buf = []
                pct = min(100, done * 100 // total)
                if pct // 10 != shown:
                    shown = pct // 10
                    print(f"   {pct:3d}%  ({done}/{total} frames, {time.time() - t0:.0f}s)", flush=True)
            if not ok:
                break
    finally:
        cap.release()
        enc.stdin.close()
        enc.wait()
    if enc.returncode != 0:
        raise RuntimeError("encoding failed")
    # 3) check the result before touching the original
    res = probe(tout)
    if res["w"] != ow or abs(res["dur"] - info["dur"]) > max(1.5, info["dur"] * 0.05):
        raise RuntimeError(f"result check failed ({res['w']}x{res['h']}, {res['dur']:.1f}s vs {info['dur']:.1f}s)")
    part = final + ".part"
    subprocess.run(["cp", tout, part], check=True)
    os.replace(part, final)
    if DELETE_ORIGINAL and os.path.exists(final) and os.path.getsize(final) > 0:
        os.remove(src)
    return final, res

def scan():
    return sorted(p for p in glob.glob(os.path.join(SRC, "*")) if os.path.isfile(p) and p.lower().endswith(VIDEO_EXT))

def watch(poll=15, quiet=False):
    """enhance every reel in the folder, then keep waiting for new ones (WATCH_MINUTES after the last one)"""
    sizes, failed, last = {}, {}, time.time()
    print(f"👀 Watching Drive › {FOLDER} … send reels from the app.", flush=True)
    while True:
        did = False
        for src in scan():
            try:
                sz = os.path.getsize(src)
            except OSError:
                continue
            if sizes.get(src) != sz or sz == 0:      # new or still arriving from the phone: wait one more round
                sizes[src] = sz
                last = time.time()
                continue
            base = os.path.splitext(os.path.basename(src))[0]
            if os.path.exists(os.path.join(OUT, base + ".mp4")):   # done before (e.g. session stopped before the delete)
                if DELETE_ORIGINAL:
                    os.remove(src)
                continue
            if failed.get(src, 0) >= 2:
                continue
            print(f"✨ {os.path.basename(src)}", flush=True)
            try:
                out, res = enhance(src)
                print(f"   ✅ saved Enhanced/{os.path.basename(out)} · {res['w']}x{res['h']} · {res['dur']:.1f}s" + (" · original deleted" if DELETE_ORIGINAL else ""), flush=True)
                did, last = True, time.time()
            except Exception as e:
                last = time.time()
                failed[src] = failed.get(src, 0) + 1
                print(f"   ⚠️ {e} (try {failed[src]}/2)", flush=True)
            finally:
                try:
                    import torch; torch.cuda.empty_cache()
                except Exception:
                    pass
        if time.time() - last > WATCH_MINUTES * 60:
            print("⏹ No new reels for a while: stopped. Run all again to start.", flush=True)
            break
        if not did:
            time.sleep(poll)

In [ ]:
#@title 3️⃣ Start (keeps running for new reels)
watch()